# 01 - Dataset Exploration & Description
**Metro Traffic Flow Prediction - Machine Learning Project**

This notebook provides an in-depth structural inspection of the Metro Traffic dataset, detailing its origin, dimensional scale, feature schema, target variable attributes, and initial data health checks.

---
### 1. Dataset Overview & Source
* **Dataset Name:** Metro Traffic Flow Operational Dataset
* **Source/Domain:** Urban Transit and Passenger Flow Analytics Repository
* **Objective:** Predict continuous metro passenger traffic flow (`Traffic_Flow_Target`) across various stations, lines, hours, and operational/weather conditions.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate project root and load dataset
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

data_path = root / "data" / "raw" / "metro_traffic_dataset_10000.csv"
if not data_path.exists():
    data_path = root / "01_Dataset" / "cleaned_dataset.csv"

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)
print(f"Dataset successfully loaded. Shape: {df.shape}")
df.head()

Loading dataset from: C:\Users\Revanth Reddy\OneDrive\Desktop\Metro Traffic -ML\data\raw\metro_traffic_dataset_10000.csv
Dataset successfully loaded. Shape: (10000, 30)


,Timestamp,Station_ID,Line_ID,Train_ID,Hour,Day_of_Week,Weekend_Flag,Holiday_Flag,Special_Event_Flag,Peak_Hour_Flag,...,Platform_Density,Temperature_C,Humidity_pct,Rainfall_mm,Visibility_km,Signal_Delay_sec,Incident_Flag,Maintenance_Flag,Energy_Consumption_kWh,Traffic_Flow_Target
0,2025-01-01 05:00:00,1,5,112,5.00,2,0,0,0,0,...,0.835,25.06,39.26,2.14,8.38,2.35,0,0,1200.0,206.15
1,2025-01-01 05:15:00,18,2,163,5.25,2,0,0,0,0,...,0.497,28.27,79.92,0.51,9.26,0.27,0,0,1200.0,248.29
2,2025-01-01 05:30:00,16,1,168,5.50,2,0,0,0,0,...,0.783,20.56,57.55,1.47,9.99,2.68,0,0,1200.0,206.25
3,2025-01-01 05:45:00,27,5,173,5.75,2,0,0,0,0,...,1.008,24.94,67.92,0.06,9.30,1.50,0,0,1200.0,256.31
4,2025-01-01 06:00:00,12,2,159,6.00,2,0,0,0,0,...,0.835,22.45,64.02,0.75,8.11,3.59,0,0,1200.0,203.10


---
### 2. Dataset Dimensions & Feature Inventory
* Total number of records (rows): `10,000`
* Total number of columns: `30`
* Features describe:
  - **Spatial identifiers:** Station_ID, Line_ID, Train_ID
  - **Temporal attributes:** Timestamp, Hour, Day_of_Week, Weekend_Flag, Holiday_Flag, Peak_Hour_Flag
  - **Passenger dynamics:** Passenger_Count, Boardings, Alightings, Ticket_Entries, Ticket_Exits, Platform_Density
  - **Transit operations:** Train_Frequency_per_Hour, Average_Dwell_Time_sec, Average_Headway_min, Average_Speed_kmph, Occupancy_Rate_pct, Signal_Delay_sec, Incident_Flag, Maintenance_Flag, Energy_Consumption_kWh
  - **Meteorological variables:** Temperature_C, Humidity_pct, Rainfall_mm, Visibility_km
  - **Target variable:** `Traffic_Flow_Target`

In [2]:
print("Dataset Information:")
df.info()

print("\nFeature Data Types Breakdown:")
print(df.dtypes.value_counts())

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Timestamp                 10000 non-null  str    
 1   Station_ID                10000 non-null  int64  
 2   Line_ID                   10000 non-null  int64  
 3   Train_ID                  10000 non-null  int64  
 4   Hour                      10000 non-null  float64
 5   Day_of_Week               10000 non-null  int64  
 6   Weekend_Flag              10000 non-null  int64  
 7   Holiday_Flag              10000 non-null  int64  
 8   Special_Event_Flag        10000 non-null  int64  
 9   Peak_Hour_Flag            10000 non-null  int64  
 10  Passenger_Count           10000 non-null  int64  
 11  Boardings                 10000 non-null  int64  
 12  Alightings                10000 non-null  int64  
 13  Ticket_Entries            10000 non-null  int64  
 1

---
### 3. Target Variable Analysis: `Traffic_Flow_Target`
The target variable is a continuous numerical measurement representing passenger traffic volume. Below are its descriptive statistics.

In [3]:
target = df["Traffic_Flow_Target"]
target_summary = pd.DataFrame({
    "Metric": ["Count", "Mean", "Std Dev", "Min", "25th Percentile", "Median (50%)", "75th Percentile", "Max", "Skewness"],
    "Value": [
        len(target),
        round(target.mean(), 2),
        round(target.std(), 2),
        round(target.min(), 2),
        round(target.quantile(0.25), 2),
        round(target.median(), 2),
        round(target.quantile(0.75), 2),
        round(target.max(), 2),
        round(target.skew(), 4)
    ]
})
display(target_summary)

,Metric,Value
0,Count,10000.0000
1,Mean,252.6600
2,Std Dev,87.0800
3,Min,45.1900
4,25th Percentile,189.9800
5,Median (50%),230.8000
6,75th Percentile,313.5100
7,Max,536.1400
8,Skewness,0.6353


---
### 4. Class Distribution Applicability
> **Note on Problem Formulation:**  
> This project addresses a **regression** problem where the objective is to predict continuous traffic volume rather than discrete classes. Therefore, a discrete categorical class distribution is not applicable. However, the target values exhibit a continuous distribution with moderate right-skewness, typical of transportation demand spikes during rush hours.

In [4]:
print("Missing Values Audit:")
missing_counts = df.isnull().sum()
print("Total Missing Values:", missing_counts.sum())

print("\nDuplicate Rows Audit:")
print("Total Duplicate Rows:", df.duplicated().sum())

Missing Values Audit:
Total Missing Values: 0

Duplicate Rows Audit:
Total Duplicate Rows: 0


---
### Key Observations from Exploration:
1. **Clean Baseline:** The dataset contains exactly 10,000 records with zero missing values and zero duplicate rows.
2. **Feature Breadth:** Contains 30 columns capturing operational, environmental, temporal, and spatial factors.
3. **Target Spread:** Traffic flow spans from `45.19` to `536.14` passengers/unit, with an average flow of `252.66`, suitable for robust regression modeling.